# Grover's search on IQM Garnet and IBM with the qBraid SDK

This notebook requires **qbraid 0.13** or later. Calibration data, best-qubit selection, the coupling map, cost estimates and compiled programs were added in 0.13. The imports cell prints the installed version.

| | |
|---|---|
| Device | IQM Garnet (`aws:iqm:qpu:garnet`), 20 superconducting qubits, paid with qBraid credits |
| Also | IBM, through your own IBM account (Part 3) |
| Cost | Garnet: 30 credits per job + 0.145 credits per shot, about 59 credits at 200 shots (100 credits = 1 US dollar). IBM jobs are billed to your IBM plan. |
| Jobs | Three on Garnet (about 177 credits at 200 shots), one on IBM |

Part 1 runs one job from start to finish. Part 2 chooses the physical qubits for a larger circuit. Part 3 runs that circuit on IBM.

## Settings

In [ ]:
DEVICE_ID = "aws:iqm:qpu:garnet"
SHOTS = 200      # measurements per job; Garnet cost = 30 + 0.145 x shots credits

# Part 1: 3 qubits, 2 iterations 
MARKED_3Q, ITERATIONS_3Q = "101", 2

# Parts 2 and 3: 4 qubits, 3 iterations 
MARKED_4Q, ITERATIONS_4Q = "1010", 3

SUBMIT = True    # True: submit new jobs from your account (3 Garnet jobs, about 177 qBraid credits at 200 shots)
JOB_IDS = {      # filled in when the jobs are submitted; paste IDs here to reload your jobs later
    "simple": None,      # Part 1, 3 qubits
    "simple_4q": None,   # Part 2, 4 qubits, default placement
    "pinned": None,      # Part 2, 4 qubits, pinned
    "ibm": None,         # Part 3, IBM
}

RUN_IBM = True   # Part 3 needs your own IBM Quantum account, with its API key in the Lab Vault; without one, skip the Part 3 cells
IBM_BACKEND = None   # None: use the least busy IBM device; or a device name such as "ibm_kingston"

In [ ]:
# Installs the packages this notebook needs if they are not already in the kernel
# Skip this cell if your environment already has them.
# When it finishes, restart the kernel (Kernel → Restart Kernel) before running the next cell.

# %pip install -q "qbraid[qiskit,visualization]==0.13.0" "qiskit==2.5.2" "qiskit-aer==0.17.2" "qiskit-ibm-runtime==0.50.0" matplotlib


In [ ]:
import qbraid
from qiskit import QuantumCircuit        # builds the circuits
from qiskit_aer import AerSimulator      # local simulator, for the ideal results
from qbraid.runtime import QbraidProvider  # access to qBraid devices and jobs

print("qbraid", qbraid.__version__)

import re                                # Python's text-pattern search; finds qubits and gates in the compiled programs

## Part 1. 

### The circuit

Grover's search finds one marked item among $2^n$ items. Each iteration flips the sign of the marked item (the oracle) and then reflects all amplitudes about their mean (the diffuser), which increases the probability of measuring the marked item.

The function below builds the circuit in Qiskit. The number of qubits is the length of `marked`.

In [ ]:
def grover(marked, iterations):
    n = len(marked)
    qc = QuantumCircuit(n)

    # 1. Equal superposition over all 2**n items
    qc.h(range(n))

    for _ in range(iterations):
        # 2. Oracle: flip the sign of the marked item.
        #    X on the qubits that are 0 in `marked` turns |marked> into |11...1>,
        #    H-MCX-H on the last qubit flips the sign of |11...1>, then the X gates are undone.
        zeros = [i for i, bit in enumerate(reversed(marked)) if bit == "0"]
        qc.x(zeros)
        qc.h(n - 1); 
        qc.mcx(list(range(n - 1)), n - 1); 
        qc.h(n - 1)
        qc.x(zeros)

        # 3. Diffuser: reflect every amplitude about the mean.
        qc.h(range(n)); 
        qc.x(range(n))
        qc.h(n - 1); 
        qc.mcx(list(range(n - 1)), n - 1); qc.h(n - 1)
        qc.x(range(n)); qc.h(range(n))

    # 4. Measure every qubit (measure_all adds a classical register named "meas")
    qc.measure_all()
    return qc


qc3 = grover(MARKED_3Q, ITERATIONS_3Q)
qc3.draw(fold=-1)

The ideal result, from a local simulator. With 2 iterations on 3 qubits the marked item is measured with probability 0.945 (random guessing: 0.125).

In [ ]:
from qiskit.visualization import plot_histogram   # counts on the y-axis

# The noiseless result
ideal3 = AerSimulator(seed_simulator=42).run(qc3, shots=SHOTS).result().get_counts()
plot_histogram(ideal3, bar_labels=False)

### The device

Get one device by its ID.

In [ ]:
provider = QbraidProvider() 

device = provider.get_device(DEVICE_ID)
device

Is it online, and how many jobs are waiting ahead of yours?

In [ ]:
print(device.status(), "| jobs waiting:", device.queue_depth())

The device profile: the program formats the device accepts directly (`program_spec`), its native gates (`basis_gates`), and its pricing. 

In [ ]:
# Fields to look at: program_spec, basis_gates, pricing
device.profile

Every circuit type `device.run()` accepts on this device. qBraid converts any of these into a format the device takes. 

In [ ]:
device.supported_run_inputs()

Estimated cost of one job, and your credit balance (100 credits = 1 US dollar).

In [ ]:
# The estimate uses the device's pricing and the shot count
print("Estimated cost:", provider.client.estimate_cost(DEVICE_ID, shots=SHOTS).estimatedCost, "credits")

# Your own credits in this organization
print("Balance:       ", provider.client.get_credits_balance()["qbraidCredits"], "credits")

### What qBraid does before submitting

By default, `device.run()` takes four steps before the job reaches the vendor:

| Step | What happens |
|---|---|
| 1. Convert | Qiskit circuit → a format the device accepts (here OpenQASM 3) |
| 2. Transform | Device-specific changes (none for qBraid-managed devices) |
| 3. Validate | Checks the circuit fits the device, e.g. qubit count, before any credits are spent |
| 4. Prepare | Packages the program for the vendor's API |
| Submit | The vendor's compiler maps the circuit to native gates and physical qubits, then the chip runs it |

`apply_runtime_profile` runs steps 1 to 4 and returns the result without submitting. The output still uses logical qubits (`q[0]`, `q[1]`, `q[2]`): qBraid does not change the gates or choose qubits. The vendor's compiler, here Amazon Braket, does that after submission.

In [ ]:
# Runs convert -> transform -> validate -> prepare and returns the payload
print(device.apply_runtime_profile(qc3).data)

### Submit

In [ ]:
if SUBMIT:
    # Run the job, minimum arguments: the circuit and number of shots
    
    job = device.run(qc3, shots=SHOTS)
    
    JOB_IDS["simple"] = job.id # save the job id for later use
    
print("Job ID:", JOB_IDS["simple"])

### Retrieve the job

Any job can be loaded again by its ID, from any notebook or session. Job IDs also appear in the **Jobs** sidebar in Lab.

In [ ]:
from qbraid.runtime import load_job

# Works with any job ID from your account
job = load_job(JOB_IDS["simple"])
job.status()

The results. `result()` waits until the job has finished.

In [ ]:
counts3 = job.result().data.get_counts()
plot_histogram(counts3, bar_labels=False)

In [ ]:
# Probability of the marked item: Garnet vs ideal simulator vs random guessing
p_garnet3 = counts3.get(MARKED_3Q, 0) / sum(counts3.values())
p_ideal3 = ideal3.get(MARKED_3Q, 0) / sum(ideal3.values())
print(f"P({MARKED_3Q}):  Garnet {p_garnet3:.3f}   ideal simulator {p_ideal3:.3f}   random {1 / 2**len(MARKED_3Q):.3f}")

plot_histogram([ideal3, counts3], legend=["ideal simulator", "Garnet"], bar_labels=False, figsize=(12, 5))

The job record: shots, cost, timing, tags and the runtime options used.

In [ ]:
# Useful keys: cost, shots, timeStamps, runtimeOptions, tags
job.metadata()

The program qBraid sent to the vendor.

In [ ]:
# Logical qubits q[...], as qBraid sent them
print(job.program().data)

The program the chip actually ran, after the vendor's compiler: native gates (`prx` and `cz` on IQM) on physical qubits, written `$N`.

In [ ]:
# Returned by the vendor: OpenQASM 3 for IQM, Quil for Rigetti direct
print(job.compiled_program().data)

The counts come back in your own bit order, with the chip placement removed, so the placement is only visible in the compiled program. Every `$N` is a physical qubit, each `b[i] = measure $N` line says which physical qubit gave classical bit `i`, and each `cz` line is one two-qubit gate.

In [ ]:
ran = job.compiled_program().data   # the compiled program as one string

# $N is a physical qubit: collect every N
print("Physical qubits:      ", sorted({int(q) for q in re.findall(r"\$(\d+)", ran)}))

# Lines like "b[0] = measure $16": classical bit 0 was read from physical qubit 16
print("Bit <- physical qubit:", re.findall(r"(\w+\[\d+\]) = measure \$(\d+)", ran))

# Each "cz " is one two-qubit gate
print("Two-qubit gates:      ", ran.count("cz "))

## Part 2. Choosing the physical qubits (4 qubits)

In Part 1 the vendor's compiler chose the physical qubits. Here the same 4-qubit circuit runs twice: once with the default placement, and once on qubits you choose. The options for choosing, and how much control they give, depend on the device.

### The chip's connections

In [ ]:
from qbraid.visualization import plot_connectivity_graph

# Nodes are qubits; edges are pairs that can run a CZ
plot_connectivity_graph(device)

### The best-calibrated qubits

`best_qubits(n)` fetches a fresh calibration and returns the connected chain of `n` qubits with the highest estimated fidelity: the product of (1 − error) over each qubit's readout and gate errors and each connection's two-qubit gate error. Calibrations change with every refresh, so choose shortly before submitting.

In [ ]:
# Returned in chain order: neighbours in the list are connected on the chip
best = device.best_qubits(4)
best

The calibration behind that choice: T1, T2, readout error and gate error for each qubit, and the CZ error on each connection in the chain.

In [ ]:
cal = device.get_calibrations()
print("Calibrated:", cal.last_calibrated)

# Per qubit: T1 and T2 in seconds; readout and gate errors are probabilities
for q in best:
    print(q, cal.qubits[str(q)])

# CZ error for each connected pair in the chosen chain
chain = {tuple(sorted(pair)) for pair in zip(best, best[1:])}
for edge in cal.edges["gateError"]["cz"]:
    if tuple(sorted((edge.source, edge.target))) in chain:
        print(edge)

### Default run

The 4-qubit circuit with 3 iterations (ideal success 0.962), submitted as is. The vendor's compiler chooses the qubits.

In [ ]:
qc4 = grover(MARKED_4Q, ITERATIONS_4Q)

if SUBMIT:
    job_4q_default = device.run(qc4, shots=SHOTS)
    JOB_IDS["simple_4q"] = job_4q_default.id
print("Job ID:", JOB_IDS["simple_4q"])

### Pin the circuit and submit

Compile the 4-qubit circuit with Qiskit onto the chosen qubits, using Garnet's connections and native gates. Then submit it with Braket's `disable_qubit_rewiring` option, which tells the vendor to keep the qubits as given.

`initial_layout` sets the qubits the circuit starts on. Qiskit's routing can still move qubits through neighbouring ones when the circuit needs connections the chosen qubits don't have.

In [ ]:
from qiskit import transpile
from qiskit.transpiler import CouplingMap

# Garnet on AWS numbers its qubits from 1 ($1 to $20); qBraid's calibration indices start at 0.
layout = [q + 1 for q in best]
edges = [[a + 1, b + 1] for a, b in device.coupling_map]
coupling = CouplingMap(edges + [[b, a] for a, b in edges])

# Use the qiskit function to transpile the circuit
pinned = transpile(
    qc4,
    coupling_map=coupling,       # only use real connections (both directions)
    initial_layout=layout,       # circuit qubit i goes to physical qubit layout[i]
    basis_gates=["r", "cz"],     # Garnet's native gates (IQM's prx is Qiskit's r)
    optimization_level=3,        # 0 to 3: how much Qiskit optimizes
)
print("Pinned to:", layout, "| gates:", dict(pinned.count_ops()))

In [ ]:
if SUBMIT:
    device.set_options(validate=False)   # qBraid counts Garnet qubits from 0 and rejects $20 (QB20); Braket accepts it
    job_pinned = device.run(
        run_input=pinned,                                   # any circuit type from supported_run_inputs()
        shots=SHOTS,
        name="grover-4q-pinned",                            # label stored with the job
        tags={"demo": "quest-grover", "layout": "pinned"},  # your own labels, for searching jobs later
        runtime_options={"disable_qubit_rewiring": True},   # passed to the vendor; keys depend on the device
        as_batch=False,                                     # True: one job for a list of circuits (only on devices that support it)
    )
    device.set_options(validate=True)    # back to the default check
    JOB_IDS["pinned"] = job_pinned.id
print("Job ID:", JOB_IDS["pinned"])

`runtime_options` is passed to the vendor. The available options, and how much control they give over placement and compilation, depend on the device. See the [QbraidProvider documentation](https://docs.qbraid.com/v2/sdk/user-guide/providers/native) and the device pages under it.

### Results: default vs pinned

`best_qubits` picks the best-calibrated chain, but a chain may not have the connections this circuit needs. The vendor's default placement uses the same calibration data and also takes the circuit into account, so it is a strong baseline.

In [ ]:
# Ideal result, same number of shots as the hardware runs
ideal4 = AerSimulator(seed_simulator=42).run(qc4, shots=SHOTS).result().get_counts()
p_ideal4 = ideal4.get(MARKED_4Q, 0) / SHOTS

# For each run: options used, qubits it ran on, two-qubit gates, probability of the marked item
counts4 = {}
for label in ("simple_4q", "pinned"):
    job4 = load_job(JOB_IDS[label])
    ran4 = job4.compiled_program().data                  # what the chip ran
    counts4[label] = job4.result().data.get_counts()     # waits until the job has finished
    qubits = sorted({int(q) for q in re.findall(r"\$(\d+)", ran4)})
    p = counts4[label].get(MARKED_4Q, 0) / sum(counts4[label].values())
    print(f"{label:10s} options {job4.metadata()['runtimeOptions']}  qubits {qubits}  two-qubit gates {ran4.count('cz ')}  P({MARKED_4Q}) {p:.3f}")
print(f"pinned layout requested {layout}   ideal P({MARKED_4Q}) {p_ideal4:.3f}   random {1 / 16:.3f}")

plot_histogram([ideal4, counts4["simple_4q"], counts4["pinned"]], legend=["ideal simulator", "default", "pinned"],
               bar_labels=False, figsize=(12, 5))

## Part 3. The same 4-qubit circuit on IBM

IBM jobs run on your own IBM account. Add your IBM Quantum API key once in the Lab **Vault** (profile menu → Vault → Add Key → IBM Quantum). These jobs are billed to your IBM plan, not qBraid credits.

On IBM the device profile lists the native gates, and IBM accepts only circuits already compiled for the device.

In [ ]:
from qbraid.runtime import QiskitRuntimeProvider

ibm_provider = QiskitRuntimeProvider()     # uses the IBM account saved in the Vault

if IBM_BACKEND:
    ibm_device = ibm_provider.get_device(IBM_BACKEND)
else:
    ibm_device = ibm_provider.least_busy(min_num_qubits=4)   # the online IBM device with the shortest queue

print("Device:", ibm_device.id)
print(ibm_device.status(), "| jobs waiting:", ibm_device.queue_depth())
ibm_device.profile                         # basis_gates is filled in on IBM

Compile with your own settings. `set_options(pass_manager=pm)` makes qBraid's transform step use this pass manager, and `apply_runtime_profile` shows the result without submitting. The result is an ISA (instruction set architecture) circuit: only the device's native gates, on physical qubits that are connected. Because IBM accepts only ISA circuits, what you see here is what runs.

In [ ]:
from qiskit.transpiler import generate_preset_pass_manager

ibm_backend = ibm_provider.runtime_service.backend(ibm_device.id)   # IBM's backend object: native gates, connections, calibration

pm = generate_preset_pass_manager(
    optimization_level=3,       # 0 to 3: how much the compiler optimizes
    backend=ibm_backend,
    seed_transpiler=7,          # fixed seed, so the preview below matches what is submitted
    # initial_layout=[...],     # optional: pin the circuit to physical qubits you choose
)
# qBraid's transform step now uses this pass manager, in the preview below and at submission
ibm_device.set_options(pass_manager=pm)

isa = ibm_device.apply_runtime_profile(qc4)   # nothing is submitted

print("Physical qubits:", isa.layout.final_index_layout())
print("Gates:          ", dict(isa.count_ops()))
print("Depth:          ", isa.depth())

Submit and retrieve.

In [ ]:
if RUN_IBM:
    # run() applies the same pass manager before submitting
    ibm_job = ibm_device.run(qc4, shots=SHOTS)
    JOB_IDS["ibm"] = ibm_job.id
    # For dynamical decoupling, submit with IBM's Sampler instead:
    # from qiskit_ibm_runtime import SamplerV2 as Sampler
    # sampler = Sampler(mode=ibm_backend)
    # sampler.options.dynamical_decoupling.enable = True
    # sampler.options.dynamical_decoupling.sequence_type = "XpXm"
    # JOB_IDS["ibm"] = sampler.run([isa], shots=SHOTS).job_id()

if JOB_IDS["ibm"]:
    ibm_job = ibm_provider.runtime_service.job(JOB_IDS["ibm"])   # IBM's own job record
    print("Status:     ", ibm_job.status())
    print("Device:     ", ibm_job.backend().name)
    print("Device time:", ibm_job.usage(), "seconds")

    # result()[0] is the first (only) circuit; .data.meas is the register created by measure_all()
    counts_ibm = ibm_job.result()[0].data.meas.get_counts()
    p_ibm = counts_ibm.get(MARKED_4Q, 0) / sum(counts_ibm.values())
    plot_histogram(counts_ibm, bar_labels=False)

The 4-qubit results side by side: ideal simulator, Garnet (default and pinned) and IBM.

In [ ]:
# Probability of the marked item for each run
for name, counts in [("Garnet default", counts4["simple_4q"]), ("Garnet pinned", counts4["pinned"]), ("IBM", counts_ibm)]:
    print(f"{name:15s} P({MARKED_4Q}) {counts.get(MARKED_4Q, 0) / sum(counts.values()):.3f}")
print(f"{'ideal':15s} P({MARKED_4Q}) {p_ideal4:.3f}   random {1 / 16:.3f}")

plot_histogram([ideal4, counts4["simple_4q"], counts4["pinned"], counts_ibm],
               legend=["ideal simulator", "Garnet default", "Garnet pinned", "IBM"], bar_labels=False, figsize=(12, 5))

## Links

- [QbraidProvider](https://docs.qbraid.com/v2/sdk/user-guide/providers/native) and the device pages under it (Amazon Braket devices, Rigetti Cepheus-1-108Q)
- [Device calibrations and connectivity](https://docs.qbraid.com/v2/sdk/user-guide/providers/native/calibrations)
- [Job execution](https://docs.qbraid.com/v2/sdk/user-guide/providers/native/jobs)
- [QiskitRuntimeProvider](https://docs.qbraid.com/v2/sdk/user-guide/providers/ibm)
- [qbraid-algorithms](https://github.com/qBraid/qbraid-algorithms): qBraid's algorithm library, which also builds Grover circuits